# Complete Synthetic PDW Dataset Generator Demo
This notebook demonstrates the generation of the final synthetic PDW dataset for Part 1.5 of the project. It includes:
1. Interleaved Radars with advanced PRI (including Dwell-and-Switch)
2. Missing Pulses & False Pulses (Noise & Multipath Echoes)
3. Kinematic Parameters (Antenna Scan Patterns & Dynamic AOA)
4. Realistic Receiver Constraints (Pulse-on-Pulse Collisions)
5. Optional Digital Quantization (disabled here by default)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from radar_generator import Environment, StableRadar, JitterRadar, StaggerRadar, SlidingRadar, DwellAndSwitchRadar

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Setting up the Complete Environment

In [ ]:
env = Environment()

# 1. Stable PRI with Frequency Hopping
# Features: Rotating antenna (scan_period=3,000,000 us = 3s), changing AOA (drift = 5 deg/s)
stable_radar = StableRadar(
    radar_id="R_STABLE", 
    pri=1000.0, 
    freq_mean=9000.0, 
    freq_mode="hopping", 
    freq_hopping_values=[9000.0, 9100.0, 9200.0],
    poi=0.90,
    multipath_prob=0.10,
    multipath_delay_mean=1.5,
    pw_mean=1.5,
    amp_mean=-55.0,
    aoa_mean=45.0,
    scan_period=3_000_000.0,  # 3 seconds rotation
    aoa_drift_rate=5.0,       # moving fast
    stretch_pw_on_multipath=False
)
env.add_radar(stable_radar)

# 2. Jitter PRI with Constant Frequency
jitter_radar = JitterRadar(
    radar_id="R_JITTER", 
    pri_mean=1500.0, 
    pri_jitter_std=50.0, 
    freq_mean=9500.0,
    poi=0.95,
    pw_mean=0.5,
    amp_mean=-50.0,
    aoa_mean=120.0,
    scan_period=0, # static staring antenna
    aoa_drift_rate=0.0
)
env.add_radar(jitter_radar)

# 3. Stagger PRI with Frequency Agility
stagger_radar = StaggerRadar(
    radar_id="R_STAGGER", 
    stagger_pris=[800.0, 1200.0, 1500.0], 
    freq_mean=8500.0, 
    freq_mode="agility", 
    freq_hopping_values=[8500.0, 8600.0, 8700.0, 8800.0],
    poi=1.0,
    pw_mean=2.0,
    amp_mean=-45.0,
    aoa_mean=210.0
)
env.add_radar(stagger_radar)

# 4. Dwell-and-Switch PRI 
dwell_radar = DwellAndSwitchRadar(
    radar_id="R_DWELL",
    dwell_pris=[700.0, 1100.0],
    pulses_per_dwell=20,
    freq_mean=10000.0,
    pw_mean=1.2,
    amp_mean=-60.0,
    aoa_mean=300.0,
    scan_period=1_500_000.0, # 1.5 second rotation
    aoa_drift_rate=-2.0
)
env.add_radar(dwell_radar)

# Generate 5,000,000 microseconds (5 seconds) of interleaved data
# Note: quantization is disabled (quantize=False) by default, simulating analog-to-digital floats
df_interleaved = env.generate(
    duration=5_000_000.0, 
    toa_noise_std=5.0, 
    freq_noise_std=2.0,
    global_drop_rate=0.02,
    noise_rate=100.0,
    quantize=False  # Enable this to snap values to receiver resolution bins
)

print(f"Total pulses generated: {len(df_interleaved)}")
display(df_interleaved.head(10))

## 2. Classic PDW Visualizations

In [ ]:
plt.figure(figsize=(15, 5))
sns.scatterplot(data=df_interleaved, x="TOA", y="Frequency", hue="Radar_ID", s=20, alpha=0.8)
plt.title("Interleaved PDW Stream: Time of Arrival vs Frequency")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Frequency (MHz)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
zoom_df = df_interleaved[df_interleaved["TOA"] < 50000]
plt.figure(figsize=(15, 5))
sns.scatterplot(data=zoom_df, x="TOA", y="Frequency", hue="Radar_ID", s=50, alpha=0.8)
plt.title("Zoomed-In: First 50ms of Interleaved Data")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Frequency (MHz)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
sns.histplot(data=df_interleaved[df_interleaved["Measured_Delta_TOA"] > 0], x="Measured_Delta_TOA", bins=100, hue="Radar_ID", multiple="stack")
plt.title("Histogram of Measured PRIs (Delta TOA)")
plt.xlabel("Delta TOA (us)")
plt.ylabel("Count")
plt.xlim(0, 3000)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
sns.scatterplot(data=df_interleaved[df_interleaved["Radar_ID"] != "NOISE"], 
                x="TOA", y="Measured_Delta_TOA", hue="Radar_ID", s=15, alpha=0.6)
plt.title("Time of Arrival vs Measured PRI (Delta TOA)")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Measured Delta TOA (us)")
plt.ylim(0, 3000)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Kinematics and Scan Pattern Visualizations

In [ ]:
plt.figure(figsize=(15, 5))
sns.scatterplot(data=df_interleaved[df_interleaved["Radar_ID"] != "NOISE"], 
                x="TOA", y="Amplitude", hue="Radar_ID", s=15, alpha=0.6)
plt.title("Amplitude Modulation due to Sinc^2 Antenna Scan Patterns")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Amplitude (dBm)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
sns.scatterplot(data=df_interleaved[df_interleaved["Radar_ID"] != "NOISE"], 
                x="TOA", y="AOA", hue="Radar_ID", s=15, alpha=0.6)
plt.title("Angle of Arrival Drift (Platform Kinematics)")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Angle of Arrival (degrees)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Reconstruct the Antenna Beam Pattern visually using a Polar Plot
radar_id = "R_STABLE"
scan_period = 3_000_000.0 # From R_STABLE configuration

plot_df = df_interleaved[df_interleaved["Radar_ID"] == radar_id].copy()

# Calculate the Emitter's internal scan angle (0 to 2pi) based on TOA
theta_rad = ((plot_df["TOA"] % scan_period) / scan_period) * 2 * np.pi

# Normalize amplitude so the weakest signal is at the center (radius 0)
min_amp = df_interleaved["Amplitude"].min()
r = plot_df["Amplitude"] - min_amp

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, projection='polar')
ax.scatter(theta_rad, r, s=15, alpha=0.7, color='blue')
ax.set_title(f"Reconstructed Antenna Scan Pattern ({radar_id})\nNormalized Amplitude vs Emitter Scan Angle", va='bottom', pad=20)
ax.set_yticklabels([]) # Hide radial ticks for a cleaner beam look
plt.tight_layout()
plt.show()

In [ ]:
# Global Polar Plot: Angle of Arrival vs Amplitude for all Radars
plt.figure(figsize=(10, 10))
ax = plt.subplot(111, projection='polar')

colors = plt.cm.tab10.colors
min_amp_global = df_interleaved["Amplitude"].min()

unique_radars = [r for r in df_interleaved['Radar_ID'].unique() if r != "NOISE"]

for i, r_id in enumerate(unique_radars):
    radar_df = df_interleaved[df_interleaved['Radar_ID'] == r_id]
    
    # Convert physical AOA to radians
    theta_rad = np.radians(radar_df['AOA'])
    
    # Normalize amplitude globally
    r = radar_df["Amplitude"] - min_amp_global
    
    ax.scatter(theta_rad, r, s=15, alpha=0.7, label=r_id, color=colors[i % len(colors)])

ax.set_title("ESM Receiver View: Angle of Arrival vs Normalized Amplitude (All Radars)", pad=20)
ax.set_theta_zero_location("N") # 0 degrees at top
ax.set_theta_direction(-1)      # Clockwise angles
ax.legend(bbox_to_anchor=(1.1, 1.05), loc='upper left')
plt.tight_layout()
plt.show()